In [1]:
import copy
import json
from collections import Counter
from dataclasses import dataclass, field
from typing import Any

from dslabs import Cluster, delay, drop, duplicate, partition
from dslabs.protocols import Message, Scheduler, Transport


In [2]:
@dataclass
class NodeTotalOrder:

    node_id: str
    peers: list[str]
    transport: Transport
    scheduler: Scheduler
    store: dict[str, Any] = field(default_factory=dict)
    log: list[tuple[str, Any]] = field(default_factory=list)

    seen_requests: set[tuple[str, int]] = field(default_factory=set)

    history: dict[int, tuple[str, Any, str, int]] = field(default_factory=dict)

    next_delivery: int = 0

    send_seq: int = 0
    pending: dict[int, tuple[str, Any]] = field(default_factory=dict)


    next_seq: int = 0
    expected: dict[str, int] = field(default_factory=dict)
    early: dict[tuple[str, int], tuple[str, Any]] = field(default_factory=dict)
    unacked: dict[int, set[str]] = field(default_factory=dict)
    retry_ms: int = 300
    retry_armed: bool = False

    @property
    def sequencer_id(self) -> str:
        return self.peers[0]


    def client_put(self, key: str, value: Any) -> None:
        send_seq = self.send_seq
        self.send_seq += 1
        self.pending[send_seq] = (key, value)
        self.send_request(key, value, send_seq)
        self.arm_retry()

    def client_get(self, key: str) -> Any:
        return self.store.get(key)


    def send_request(self, key: str, value: Any, send_seq: int, direct: bool = False) -> None:
        if self.node_id == self.sequencer_id:
            self.receive_request(key, value, self.node_id, send_seq)
            return
        msg = {"type": "request", "key": key, "value": value,
               "origin": self.node_id, "send_seq": send_seq}
        if direct:
            self.transport.send(self.sequencer_id, msg)
        else:
            self.broadcast(msg)

    def receive_request(self, key: str, value: Any, origin: str, send_seq: int) -> None:
        if send_seq < self.expected.get(origin, 0):
            return
        self.early[(origin, send_seq)] = (key, value)
        while (origin, self.expected.get(origin, 0)) in self.early:
            at = self.expected.get(origin, 0)
            key, value = self.early.pop((origin, at))
            self.expected[origin] = at + 1
            self.assign(key, value, origin, at)
        self.arm_retry()

    def assign(self, key: str, value: Any, origin: str, send_seq: int) -> None:
        seq = self.next_seq
        self.next_seq += 1
        self.unacked[seq] = {p for p in self.peers if p != self.node_id}
        self.receive_ordered(key, value, seq, origin, send_seq)
        for peer in self.peers:
            if peer != self.node_id:
                self.send_ordered(peer, seq)

    def send_ordered(self, to: str, seq: int) -> None:
        key, value, origin, send_seq = self.history[seq]
        self.transport.send(to, {"type": "ordered", "key": key, "value": value,
                                 "seq": seq, "origin": origin, "send_seq": send_seq})

    def receive_ordered(self, key: str, value: Any, seq: int, origin: str, send_seq: int) -> None:
        if seq in self.history:
            return
        self.history[seq] = (key, value, origin, send_seq)
        while self.next_delivery in self.history:
            k, v, who, at = self.history[self.next_delivery]
            self.deliver(k, v)
            if who == self.node_id:
                self.pending.pop(at, None)
            self.next_delivery += 1

    def broadcast(self, msg: Message) -> None:
        for peer in self.peers:
            if peer != self.node_id:
                self.transport.send(peer, msg)

    def deliver(self, key: str, value: Any) -> None:
        self.store[key] = value
        self.log.append((key, value))


    def arm_retry(self) -> None:
        if not self.retry_armed and (self.pending or self.unacked):
            self.retry_armed = True
            self.scheduler.call_later(self.retry_ms, self.retry)

    def retry(self) -> None:
        self.retry_armed = False
        for send_seq, (key, value) in list(self.pending.items()):

            self.send_request(key, value, send_seq, direct=True)
        for seq, waiting in list(self.unacked.items()):
            for peer in self.peers:
                if peer in waiting:
                    self.send_ordered(peer, seq)
        self.arm_retry()


    def on_message(self, msg: Message) -> None:
        if msg["type"] == "request":
            if self.node_id == self.sequencer_id:
                self.receive_request(msg["key"], msg["value"], msg["origin"], msg["send_seq"])
            elif (msg["origin"], msg["send_seq"]) not in self.seen_requests:
                self.seen_requests.add((msg["origin"], msg["send_seq"]))
                self.broadcast(msg)
        elif msg["type"] == "ordered":


            self.transport.send(msg["from"], {"type": "ack", "seq": msg["seq"]})
            self.receive_ordered(msg["key"], msg["value"], msg["seq"], msg["origin"], msg["send_seq"])
        elif msg["type"] == "ack":
            waiting = self.unacked.get(msg["seq"])
            if waiting is not None:
                waiting.discard(msg["from"])
                if not waiting:
                    del self.unacked[msg["seq"]]
        else:
            raise ValueError(f"Unknown message type in {msg!r}")


    def brief_state(self) -> dict[str, Any]:
        return dict(self.store)


In [3]:
@dataclass
class NodeJobs(NodeTotalOrder):
    """CAS and jobs over TOB. Job code defines run(item).

    Handles network failures with nodes alive. The log is in memory;
    permanent crashes and recovery of Python side effects are not handled.
    """

    jobs: dict[str, dict] = field(default_factory=dict)
    registers: dict[str, Any] = field(default_factory=dict)
    replies: dict[str, dict] = field(default_factory=dict)
    completions: list[tuple[str, int, Any]] = field(default_factory=list)
    executions: dict[tuple[str, int], int] = field(default_factory=dict)
    api_counter: int = 0

    def cas(self, key: str, expected: Any, value: Any) -> bool:
        if self.registers.get(key) != expected:
            return False
        self.registers[key] = copy.deepcopy(value)
        return True

    def propose(self, command: dict) -> None:

        self.scheduler.call_later(0, lambda: self.client_put("__jobs__", command))

    def on_message(self, msg: dict) -> None:
        if msg["type"] in {"submit_job", "query_job_status", "get_job_results", "cas"}:
            command = {k: copy.deepcopy(v) for k, v in msg.items() if k != "from"}
            self.api_counter += 1
            command.setdefault("request_id", f"api:{self.node_id}:{self.api_counter}")
            command["reply_to"] = msg["from"]
            command["ingress"] = self.node_id
            self.propose(command)
        else:
            super().on_message(msg)

    def assign(self, key, value, origin, send_seq):
        super().assign(key, value, origin, send_seq)

        self.unacked = {seq: waiting for seq, waiting in self.unacked.items() if waiting}

    def deliver(self, key: str, value: Any) -> None:
        if key != "__jobs__":
            super().deliver(key, value)
            return
        command = copy.deepcopy(value)

        self.log.append((key, command))
        kind = command["type"]
        request_id = command.get("request_id")
        if request_id is not None:
            if request_id not in self.replies:
                self.replies[request_id] = self.apply_api(command)
            if command["ingress"] == self.node_id:
                reply = dict(self.replies[request_id], request_id=request_id)
                self.transport.send(command["reply_to"], reply)
        elif kind == "claim":
            job_id, index = command["job_id"], command["index"]
            task = self.jobs[job_id]["tasks"][index]
            won = self.cas(task["owner_key"], None, command["worker"])
            if won and command["worker"] == self.node_id:
                self.scheduler.call_later(0, lambda: self.execute(job_id, index))
        elif kind == "finish":
            job_id, index = command["job_id"], command["index"]
            task = self.jobs[job_id]["tasks"][index]
            if not task["done"] and self.registers.get(task["owner_key"]) == command["worker"]:
                task["done"] = True
                task["result"] = command["result"]
                self.completions.append((job_id, index, command["result"]))

    def apply_api(self, command: dict) -> dict:
        kind = command["type"]
        if kind == "cas":

            key = "user:" + command["key"]
            ok = self.cas(key, command["expected"], command["value"])
            return {"type": "cas_response", "swapped": ok,
                    "value": copy.deepcopy(self.registers.get(key))}
        job_id = command["job_id"]
        job = self.jobs.get(job_id)
        response = {"type": kind + "_response", "job_id": job_id}
        if kind == "submit_job":
            action, data = command.get("job_action"), command.get("job_data")
            try:
                if not isinstance(action, str) or not isinstance(data, list):
                    raise ValueError("Expected Python source and a list of inputs")
                compile(action, "<job>", "exec")
                json.dumps(data, allow_nan=False)
                if job is not None and (job["action"] != action or job["data"] != data):
                    raise ValueError("job_id already used for a different job")
                if job is None:
                    self.jobs[job_id] = {"action": action, "data": data, "tasks": [
                        {"owner_key": "task:" + json.dumps([job_id, i]), "done": False}
                        for i in range(len(data))]}
                    for index in range(len(data)):
                        self.propose({"type": "claim", "job_id": job_id,
                                      "index": index, "worker": self.node_id})
                response["job_submitted"] = True
            except (ValueError, TypeError, SyntaxError) as error:
                response.update(job_submitted=False, error=str(error))
        elif kind == "query_job_status":
            response["job_status"] = ("not_found" if job is None else
                                      "complete" if all(t["done"] for t in job["tasks"]) else
                                      "in_progress")
        elif kind == "get_job_results":
            complete = job is not None and all(t["done"] for t in job["tasks"])
            response["job_results"] = ([copy.deepcopy(t["result"]) for t in job["tasks"]]
                                       if complete else None)
        return response

    def execute(self, job_id: str, index: int) -> None:
        job = self.jobs[job_id]
        identity = (job_id, index)
        self.executions[identity] = self.executions.get(identity, 0) + 1
        try:
            namespace = {}
            exec(job["action"], namespace)
            result = namespace["run"](copy.deepcopy(job["data"][index]))

            result = json.loads(json.dumps(result, allow_nan=False))
        except Exception as error:
            result = {"error": type(error).__name__, "message": str(error)}
        self.propose({"type": "finish", "job_id": job_id, "index": index,
                      "worker": self.node_id, "result": result})


In [4]:
class JobClient:
    def __init__(self, cluster, client_id="client", retry_ms=500):
        self.cluster = cluster
        self.client_id = client_id
        self.retry_ms = retry_ms
        self.pending = {}
        self.responses = {}
        self.counter = 0
        cluster.network.register(client_id, self.on_message)
        self.transport = cluster.network.endpoint(client_id)

    def request(self, node_id, message):
        self.counter += 1
        request_id = f"{self.client_id}:{self.counter}"
        message = dict(copy.deepcopy(message), request_id=request_id)
        self.pending[request_id] = (node_id, message)
        self.send(request_id)
        return request_id

    def send(self, request_id):
        if request_id in self.pending:
            node, message = self.pending[request_id]
            self.transport.send(node, message)
            self.cluster.scheduler.call_later(self.retry_ms, lambda: self.send(request_id))

    def on_message(self, message):
        request_id = message["request_id"]
        self.responses.setdefault(request_id, message)
        self.pending.pop(request_id, None)

    def wait(self, request_id, timeout_ms=60000):
        deadline = self.cluster.scheduler.now_ms() + timeout_ms
        while request_id not in self.responses:
            event = self.cluster.peek()
            if event is None or event.due_ms > deadline:
                raise TimeoutError(request_id)
            self.cluster.step()
        return self.responses[request_id]


In [5]:
cluster = Cluster(NodeJobs, 3, seed=7)
cluster.add_rule(drop(0.3))
cluster.add_rule(duplicate(0.3))
client = JobClient(cluster)


In [6]:
requests = [client.request(node, {
    "type": "cas", "key": "lock", "expected": None, "value": node
}) for node in cluster.node_ids]
replies = [client.wait(request) for request in requests]
assert sum(reply["swapped"] for reply in replies) == 1
[(reply["swapped"], reply["value"]) for reply in replies]


[(True, 'n1'), (False, 'n1'), (False, 'n1')]

In [7]:
action = """def run(item):
    lines = item.splitlines()
    return {"lines": len(lines), "errors": [line for line in lines if "ERROR" in line]}
"""
logs = [
    "INFO started\nERROR connection lost\nINFO retry",
    "INFO started\nINFO finished",
    "ERROR invalid input\nERROR request failed",
]
request = client.request("n2", {
    "type": "submit_job", "job_id": "logs-1",
    "job_action": action, "job_data": logs,
})
response = client.wait(request)
assert response["job_submitted"]
response


{'type': 'submit_job_response',
 'job_id': 'logs-1',
 'job_submitted': True,
 'request_id': 'client:4',
 'from': 'n2'}

In [8]:
request = client.request("n3", {"type": "query_job_status", "job_id": "logs-1"})
response = client.wait(request)
assert response["job_status"] != "not_found"
response


{'type': 'query_job_status_response',
 'job_id': 'logs-1',
 'job_status': 'complete',
 'request_id': 'client:5',
 'from': 'n3'}

In [9]:
assert cluster.run_until_idle(cluster.scheduler.now_ms() + 60000)
request = client.request("n1", {"type": "get_job_results", "job_id": "logs-1"})
results = client.wait(request)["job_results"]
assert [result["lines"] for result in results] == [3, 2, 2]
results


[{'lines': 3, 'errors': ['ERROR connection lost']},
 {'lines': 2, 'errors': []},
 {'lines': 2, 'errors': ['ERROR invalid input', 'ERROR request failed']}]

In [10]:
assert cluster.run_until_idle(cluster.scheduler.now_ms() + 60000)
executions = Counter()
for node in cluster.nodes.values():
    executions.update(node.executions)
    assert len(node.completions) == len(logs)
assert executions == Counter({("logs-1", i): 1 for i in range(len(logs))})
print("Executions:", dict(executions))
print("Completions per replica:", {n: len(cluster[n].completions) for n in cluster.node_ids})


Executions: {('logs-1', 0): 1, ('logs-1', 1): 1, ('logs-1', 2): 1}
Completions per replica: {'n1': 3, 'n2': 3, 'n3': 3}


In [11]:
ACTION = "def run(item):\n    return item * item"


def setup(nodes=3, seed=0):
    cluster = Cluster(NodeJobs, nodes, seed=seed)
    return cluster, JobClient(cluster)


def submit(client, node="n2", job_id="squares", data=None, action=ACTION):
    return client.request(node, {"type": "submit_job", "job_id": job_id,
                                 "job_action": action, "job_data": [2, 3, 4] if data is None else data})


def assert_finished(cluster, expected):
    assert cluster.run_until_idle(cluster.scheduler.now_ms() + 60000)
    counts = Counter()
    for node in cluster.nodes.values():
        counts.update(node.executions)
        assert [task["result"] for task in node.jobs["squares"]["tasks"]] == expected
        assert len(node.completions) == len(expected)
        assert len({(j, i) for j, i, _ in node.completions}) == len(expected)
    assert counts == Counter({("squares", i): 1 for i in range(len(expected))})
    logs = [node.log for node in cluster.nodes.values()]
    assert all(log == logs[0] for log in logs)


In [12]:
def test_cas_has_one_winner_and_retries_do_not_apply_twice():
    cluster, client = setup()
    cluster.add_rule(duplicate(1))
    ids = [client.request(n, {"type": "cas", "key": "lock", "expected": None, "value": n})
           for n in cluster.node_ids]
    replies = [client.wait(r) for r in ids]
    assert sum(r["swapped"] for r in replies) == 1
    assert cluster.run_until_idle()
    assert len({n.registers["user:lock"] for n in cluster.nodes.values()}) == 1


In [13]:
def test_jobs_with_loss_duplicates_and_delay(seed):
    cluster, client = setup(seed=seed)
    cluster.add_rule(drop(0.5))
    cluster.add_rule(duplicate(0.5))
    cluster.add_rule(delay(0, 200))
    request = submit(client)
    assert client.wait(request)["job_submitted"] is True
    assert_finished(cluster, [4, 9, 16])
    result = client.request("n3", {"type": "get_job_results", "job_id": "squares"})
    assert client.wait(result)["job_results"] == [4, 9, 16]


In [14]:
def test_reads_wait_for_ordered_state_after_submission():
    cluster, client = setup()
    cut = partition({"n3"})
    cluster.add_rule(cut)
    assert client.wait(submit(client))["job_submitted"]
    assert "squares" not in cluster["n3"].jobs
    request = client.request("n3", {"type": "query_job_status", "job_id": "squares"})
    cluster.run_until(cluster.scheduler.now_ms() + 1500)
    assert request not in client.responses
    cluster.remove_rule(cut)
    assert client.wait(request)["job_status"] in {"in_progress", "complete"}
    assert_finished(cluster, [4, 9, 16])


In [15]:
def test_submission_waits_when_sequencer_is_partitioned():
    cluster, client = setup()
    cut = partition({"n1"})
    cluster.add_rule(cut)
    request = submit(client)
    cluster.run_until(1500)
    assert request not in client.responses
    cluster.remove_rule(cut)
    assert client.wait(request)["job_submitted"]
    assert_finished(cluster, [4, 9, 16])


In [16]:
def test_repeated_job_id_and_conflicting_submission():
    cluster, client = setup()
    ids = [submit(client, node=n) for n in cluster.node_ids]
    assert all(client.wait(r)["job_submitted"] for r in ids)
    conflict = submit(client, data=[99])
    assert client.wait(conflict)["job_submitted"] is False
    assert_finished(cluster, [4, 9, 16])


In [17]:
def test_unknown_empty_and_failed_jobs():
    cluster, client = setup()
    query = client.request("n3", {"type": "query_job_status", "job_id": "missing"})
    assert client.wait(query)["job_status"] == "not_found"
    assert client.wait(submit(client, data=[]))["job_submitted"]
    status = client.request("n1", {"type": "query_job_status", "job_id": "squares"})
    assert client.wait(status)["job_status"] == "complete"
    bad = submit(client, job_id="error", action="def run(item):\n    raise ValueError('bad input')")
    assert client.wait(bad)["job_submitted"]
    assert cluster.run_until_idle()
    results = client.request("n2", {"type": "get_job_results", "job_id": "error"})
    assert all(r["error"] == "ValueError" for r in client.wait(results)["job_results"])


In [18]:
def test_one_node_cluster_goes_idle():
    cluster, client = setup(nodes=1)
    assert client.wait(submit(client, node="n1"))["job_submitted"]
    assert_finished(cluster, [4, 9, 16])


In [19]:
def test_lost_client_responses_and_identical_inputs():
    cluster, client = setup()
    def lose_responses(frm, to, deliveries, rng):
        return [] if to == "client" else deliveries
    cluster.add_rule(lose_responses)
    request = submit(client, data=[2, 2, 2])
    cluster.run_until(2000)
    assert request not in client.responses
    cluster.remove_rule(lose_responses)
    assert client.wait(request)["job_submitted"]
    assert_finished(cluster, [4, 4, 4])


In [20]:
def test_original_message_schema_without_request_id():
    cluster, client = setup()
    received = []
    cluster.network.register("raw_client", received.append)
    cluster.network.endpoint("raw_client").send("n2", {
        "type": "submit_job", "job_id": "squares", "job_action": ACTION, "job_data": [2, 3, 4]})
    assert_finished(cluster, [4, 9, 16])
    assert received[0]["job_submitted"]


In [21]:
tests = [(name, fn) for name, fn in list(globals().items())
         if name.startswith("test_") and callable(fn)]
passed = 0
for name, test in tests:
    if name == "test_jobs_with_loss_duplicates_and_delay":
        for seed in (0, 7, 42):
            test(seed)
            passed += 1
    else:
        test()
        passed += 1
print(f"{passed} tests passed")


11 tests passed
